# 🎬 TH-Labs AI Dubbing — Colab (free T4 GPU) runner

Runs the **full** pipeline — Whisper · NLLB-200 · **OmniVoice** zero-shot voice cloning · Demucs — on Colab's free **T4 GPU**, and serves the Studio UI on a **public URL** via a Cloudflare quick tunnel (no account, no ngrok token).

The React app is built to static files and served by the FastAPI backend, so the whole thing is **one port behind one tunnel**.

### Why OmniVoice works here (and not on a 6 GB laptop)
OmniVoice is **~3.1 GB on disk and ~2.2 GB of VRAM in fp16** — it fits a T4 easily, alongside Whisper-medium and NLLB (≈8 GB of the T4's 16 GB total). The reason it can't run on the dev laptop was never VRAM: it needs **`transformers>=5.3`**, which is unreachable on Python 3.14 because the newer Rust `tokenizers` segfaults there. Colab runs Python 3.12, so the constraint disappears.

> ⚠️ **Ephemeral.** Colab ends the session on idle (~90 min) or after ~12 h, and you get a **new URL** each run. For an always-on link use Modal or Hugging Face Spaces instead.

---
### Before you run
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Push your latest code to GitHub.
3. Run every cell top-to-bottom. Cell 6 is a **preflight** that fails loudly if OmniVoice or NLLB is broken — check it before waiting on the build. The **last cell prints your public URL**.

In [ ]:
# 1 · Confirm a GPU is attached.  Errors here → Runtime → Change runtime type → T4 GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys; print("python", sys.version.split()[0], "(omnivoice needs >=3.10)")

In [ ]:
# 2 · Clone the repo.  It's PRIVATE, so paste a GitHub token (scope: repo) when
#     prompted.  If you've made the repo public, just press Enter to skip the token.
import getpass, os, subprocess

REPO   = "asilbekali/TH-Labs-full"   # owner/name
BRANCH = "main"
DEST   = "/content/TH-Labs-full"

token = getpass.getpass(f"GitHub token for {REPO} (Enter if public): ").strip()
url   = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
subprocess.run(["rm", "-rf", DEST], check=False)
rc = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, DEST]).returncode
del token, url          # don't keep the token around
assert rc == 0, "clone failed — check the token / repo name / branch"
os.chdir(DEST)
print("✔ cloned into", DEST)

In [ ]:
# 3 · System packages: ffmpeg + Node (usually already on Colab — this just ensures it).
!ffmpeg -version >/dev/null 2>&1 && echo "ffmpeg ✔" || (apt-get -qq update && apt-get -qq install -y ffmpeg)
!node --version >/dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get -qq install -y nodejs)
!echo "node $(node --version) · npm $(npm --version)"

In [ ]:
# 4 · Python ML stack.  Colab ships CUDA torch already; we keep it.
#     First run ~3–5 min.
%pip install -q -r backend/requirements.txt
%pip install -q openai-whisper sentencepiece edge-tts soundfile silero-vad demucs huggingface_hub

In [ ]:
# 5 · OmniVoice — the voice-cloning engine.
#
# TWO GOTCHAS, both learned the hard way:
#   (a) omnivoice requires torchaudio; if it's missing the resolver backtracks
#       and tries to build an ANCIENT numba (0.53.1) that cannot compile on
#       Python >=3.10.  Install torchaudio FIRST.
#   (b) pin `numba>=0.61` for the same reason — it forces a modern wheel
#       instead of a source build that fails.
# Installing omnivoice upgrades transformers to 5.x, which is exactly what we
# want here (and what Python 3.14 could not have).
%pip install -q torchaudio
%pip install -q omnivoice "numba>=0.61" "librosa>=0.11"

import transformers, torch
print(f"✔ transformers {transformers.__version__} (need >=5.3) · torch {torch.__version__} · CUDA {torch.cuda.is_available()}")

In [ ]:
# 6 · PREFLIGHT — fail fast, before spending minutes on the frontend build.
#     Checks the three things that actually break: OmniVoice import, the
#     language mapping for our targets, and NLLB under transformers 5.x.
import sys, torch
ok = True

# (a) OmniVoice imports?  This is what fails on transformers 4.x.
try:
    from omnivoice import OmniVoice
    print("✔ omnivoice imports")
except Exception as e:
    ok = False; print(f"[X] omnivoice import FAILED: {type(e).__name__}: {e}")

# (b) language mapping for Uzbek / Russian / English
sys.path.insert(0, "backend")
try:
    from app.pipeline.tts import resolve_language
    import omnivoice, os, re
    lm = os.path.join(os.path.dirname(omnivoice.__file__), "utils", "lang_map.py")
    pairs = dict(re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', open(lm, encoding="utf-8").read()))
    valid = set(pairs) | set(pairs.values())
    for c in ("uz", "ru", "en"):
        r = resolve_language(c)
        flag = "✔" if r in valid else "✗"
        if r not in valid: ok = False
        print(f"  {flag} {c} -> {r!r}")
except Exception as e:
    ok = False; print(f"✗ language check FAILED: {type(e).__name__}: {e}")

# (c) NLLB still works on transformers 5.x?  (the real upgrade risk)
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    mid = "facebook/nllb-200-distilled-600M"
    tok = AutoTokenizer.from_pretrained(mid, src_lang="eng_Latn")
    mdl = AutoModelForSeq2SeqLM.from_pretrained(mid).to("cuda").eval()
    bid = tok.convert_tokens_to_ids("uzn_Latn")
    enc = tok("Today we translate video with artificial intelligence.", return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = mdl.generate(**enc, forced_bos_token_id=bid, max_new_tokens=48)
    print("✔ NLLB EN→UZ:", tok.batch_decode(out, skip_special_tokens=True)[0])
    del mdl; torch.cuda.empty_cache()
except Exception as e:
    ok = False; print(f"✗ NLLB FAILED: {type(e).__name__}: {e}")

print("\n" + ("✅ PREFLIGHT PASSED — continue." if ok else "❌ PREFLIGHT FAILED — fix before continuing."))

In [ ]:
# 7 · Build the React UI → frontend/dist (FastAPI serves it).  ~1–2 min.
import os, subprocess
subprocess.run("npm ci --no-audit --no-fund || npm install --no-audit --no-fund",
               cwd="frontend", shell=True, check=True)
subprocess.run("npm run build", cwd="frontend", shell=True, check=True)
print("✔ frontend built →", os.path.exists("frontend/dist/index.html"))

In [ ]:
# 8 · Cloudflare quick-tunnel binary (no account needed).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared && cloudflared --version

In [ ]:
# 9 · Launch the GPU backend + tunnel.  KEEP THIS CELL RUNNING (and the tab open).
import os, re, sys, time, subprocess, threading, urllib.request

os.environ.update({
    "TH_LABS_MODE":              "auto",     # real models where installed
    "TH_LABS_WHISPER_MODEL":     "medium",   # paper-grade; a T4 handles it
    "TH_LABS_WHISPER_DEVICE":    "cuda",
    "TH_LABS_OMNIVOICE_DEVICE":  "cuda:0",   # OmniVoice cloning on GPU (~2.2 GB)
    "TH_LABS_SEPARATION_DEVICE": "cuda",     # Demucs on GPU
    "TH_LABS_CLONE_DEVICE":      "cuda",     # OpenVoice fallback, if installed
})

# FastAPI (no --reload → one clean process)
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=lambda: [print("[api]", ln, end="") for ln in api.stdout], daemon=True).start()

for _ in range(90):                          # wait for the port to bind
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("backend did not start — scroll up for [api] errors")

# confirm OmniVoice really is the TTS engine (not a silent edge-tts fallback)
import json
h = json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health"))
tts = next(s for s in h["stages"] if s["key"] == "tts")
print(f"✔ backend up · TTS engine = {tts['engine']} ({tts['mode']})")
if "omni" not in tts["engine"].lower():
    print("  ⚠ OmniVoice did NOT load — the pipeline will fall back to edge-tts.")

# Cloudflare tunnel → grab the public URL
tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public = None
for ln in tun.stdout:
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", ln)
    if m:
        public = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()  # drain

print("\n" + "=" * 70)
print("  🎬  OPEN YOUR DUBBING STUDIO:")
print("      " + str(public))
print("=" * 70)
print("\nFirst dub downloads the models (~6 GB, a few minutes). Leave this cell running.")

## Language configuration (Uzbek · Russian · English)

OmniVoice covers **644 languages** and `generate()` takes a `language=` argument — its docs note quality is **better than language-agnostic mode**, and for Uzbek the label matters: unlabelled Uzbek text can be voiced with a neighbouring language's phonetics.

`backend/app/pipeline/tts.py` maps the app's code to an OmniVoice identifier via `OMNIVOICE_LANG` / `resolve_language()`:

| App | Sent to OmniVoice | Note |
|-----|-------------------|------|
| `uz` Uzbek   | `uz`  | NLLB emits `uzn_Latn`; `uzn` (Northern Uzbek) is also valid if you want the narrower variant |
| `ru` Russian | `ru`  | Cyrillic, matches NLLB's `rus_Cyrl` |
| `en` English | `en`  | — |
| `ar` Arabic  | `arb` | **exception** — OmniVoice has no generic `ar`, only variants |

All 32 app languages resolve; `auto` → `None` (language-agnostic) rather than a bogus code. Each segment is also generated with `duration=` set to its source slot, so dubbed speech fits the original timing natively instead of being time-stretched by ffmpeg afterwards.

## Using it
- Open the URL, try the **built-in sample** or upload a clip.
- **Quality:** *Fast* is quickest; *Studio* (Whisper-medium + Demucs + OmniVoice) is best — the T4 makes it practical.
- Check the navbar badge and `/api/health` to confirm the TTS engine reads **OmniVoice**, not edge-tts.

## Keeping it alive / stopping
- Keep the tab open; Colab reclaims idle GPUs.
- Stop with **Runtime → Interrupt**, or `!pkill -f cloudflared; pkill -f uvicorn`.
- Re-running the launch cell issues a **fresh URL**.